# Crossing Watch: letter and fix by Muse Glimmer 30B (open weights)

SAM 3.1 does the tracking and the maths elsewhere; here Meta's open-weights **Muse Glimmer 30B** (official 4-bit GGUF,
Apache 2.0) runs on Kaggle's free **2 x T4** and writes the fix and the letter from the measured numbers, looking at
the blurred street photo when the pack has one. Every number in the letter is still checked by code against the
measurements.

**Settings → Accelerator: GPU T4 x2. Internet: on.** Add the `crossing-watch` dataset with **Add Input**: a zip of
one commit of the repo, made with

    git archive --format=zip --add-virtual-file=GIT_SHA:$(git rev-parse HEAD) -o crossing-watch.zip HEAD

so the run can record which code it ran. Optional: a Kaggle secret `MODEL_API_KEY` lets Muse Spark review Glimmer's
letter and Muse Image draw the fix.

Add our gate's measurements too: the private `cw-gate` dataset (`events.json` and `site.json` only). The notebook
pins llama.cpp `b11976` and the model's Hugging Face revision, checks both files against their published sha256,
runs one 32,768-token slot with medium reasoning and the model card's sampling, stops with a clear error if Glimmer
runs out of context, saves the letter (`glimmer-letter.txt`) and `plan.json` as outputs, and writes the run record
(`glimmer-gate-provenance.json`).

The plan is sent with a JSON schema that gives its shape only. The number lock (`LETTER_MODEL_NUMBER_LOCK=1`, digits
only as the measured numbers) stays off: on Fireworks it trapped Muse Spark Open in "a0a0a0..." text. A reply that is
wordless or one short piece repeated goes back as unreadable, the code check and the reviewer still run, and the
run record counts how many tries Glimmer needed.

The Kaggle notebook was run by the author on the author's own Kaggle account.

In [ ]:
import json, subprocess, time
t0 = time.time()  # the provenance file gives each step's end in seconds from here
NVIDIA_SMI = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout
GPUS = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True,
                      check=True).stdout.splitlines()
HOST = f"Kaggle {len(GPUS)} x {GPUS[0].removeprefix('Tesla ')}"  # e.g. "Kaggle 2 x T4", named under the letter
print(NVIDIA_SMI, HOST)

## 1. Build llama.cpp's server for the T4 (about 28 minutes)

In [ ]:
%%bash
set -eo pipefail
cd /kaggle/working
# a release tag, not master: master can change under an unattended run
[ -d llama.cpp ] || git clone --depth 1 --branch b11976 https://github.com/ggml-org/llama.cpp
# NO_VMM: Kaggle's image has no CUDA driver stub for CMake to link, and VMM is optional
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF | tail -3
cmake --build llama.cpp/build --config Release -j 4 --target llama-server 2>&1 | tail -2

## 2. Download Meta's official 4-bit Glimmer (16.8 GB) and its image encoder (1.4 GB), and check both against Hugging Face's published sha256

In [ ]:
import hashlib
from huggingface_hub import hf_hub_download
marks = {'built': round(time.time() - t0)}
REPO = 'meta-models/Muse-Glimmer-30B-GGUF'
REVISION = 'b44f1072cf75e918a1b82a3a17b45c4389b076e7'  # the repo's commit whose files have these hashes
SHA256 = {'Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf': '4cc57c0f51040a226e5a72cc47b7613f7772950e460a665f7083de89f183f60e',
          'mmproj-Muse-Glimmer-30B-Q4_K_M.gguf': 'f48b452316f9b213758e8659444029b961a24a07f99a1abb2a9f88b06f7c00c6'}


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()


paths = {name: hf_hub_download(REPO, name, revision=REVISION, local_dir='/tmp/glimmer') for name in SHA256}
for name, path in paths.items():
    assert sha256(path) == SHA256[name], f'{name} is not the file Hugging Face published'
model, mmproj = paths['Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf'], paths['mmproj-Muse-Glimmer-30B-Q4_K_M.gguf']
marks['downloaded'] = round(time.time() - t0)
model, mmproj

## 3. Start the server on both GPUs: one 32,768-token slot, medium reasoning, the model card's sampling

In [ ]:
import urllib.request
LOG = '/kaggle/working/server.log'
ARGS = ['-m', model, '--mmproj', mmproj, '-ngl', '99', '--split-mode', 'layer', '--jinja', '--port', '8080',
        '-np', '1',  # one slot gets all of -c (the model card: "a single request gets -c / -np")
        # medium, not the template's default "high": thinking that fills the context leaves no answer at all
        '--chat-template-kwargs', json.dumps({'reasoning_strength': 'medium'}),
        # the card's sampling; it sets no min_p (llama-server's own defaults: temp 0.8, top_k 40, min_p 0.05)
        '--temp', '1.0', '--top-p', '0.95', '--top-k', '64', '--min-p', '0']


def start(context):
    """llama-server with one slot of `context` tokens, or None if it exits (e.g. out of GPU memory) or never gets healthy."""
    proc = subprocess.Popen(['/kaggle/working/llama.cpp/build/bin/llama-server', *ARGS, '-c', str(context)],
                            stdout=open(LOG, 'a'), stderr=subprocess.STDOUT)
    for _ in range(120):
        if proc.poll() is not None:
            return None
        try:
            if urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=5).status == 200:
                return proc
        except OSError:  # refused while starting, 503 while loading
            time.sleep(10)
    proc.kill()
    return None


for CONTEXT in (32768, 16384):  # 16384 only if 32768 does not fit on the two T4s
    server = start(CONTEXT)
    if server:
        break
else:
    raise RuntimeError('llama-server did not start:\n' + open(LOG).read()[-3000:])
PROPS = json.load(urllib.request.urlopen('http://127.0.0.1:8080/props', timeout=30))
marks['server_ready'] = round(time.time() - t0)
print('server ready:', PROPS.get('build_info'), f'-c {CONTEXT}')

## 4. Write the evidence pack with Glimmer

In [ ]:
import glob, os, shutil
src = os.path.dirname(glob.glob('/kaggle/input/**/pack.py', recursive=True)[0])
CW = '/kaggle/working/cw'
shutil.copytree(src, CW, dirs_exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['MODEL_API_KEY'] = UserSecretsClient().get_secret('MODEL_API_KEY')
except Exception:
    print('no MODEL_API_KEY secret: Glimmer writes the letter, no Spark review, Muse Image is skipped')
OUT = f'{CW}/gate'  # our gate's measurements (events.json, site.json) from the private cw-gate dataset
os.makedirs(OUT, exist_ok=True)
[shutil.copy(p, OUT) for p in glob.glob('/kaggle/input/**/cw-gate/*.json', recursive=True)]
assert os.path.exists(f'{OUT}/events.json') and os.path.exists(f'{OUT}/site.json'), 'add the cw-gate dataset'
for stale in ('plan.json', 'street.jpg'):  # both must come from this run
    if os.path.exists(f'{OUT}/{stale}'):
        os.remove(f'{OUT}/{stale}')
env = {**os.environ, 'LETTER_MODEL_URL': 'http://127.0.0.1:8080/v1', 'LETTER_MODEL_HOST': HOST}
started = time.time()
run = subprocess.run(['python3', 'pack.py', OUT, f'{OUT}/site.json'], cwd=CW, env=env, capture_output=True, text=True)
pack_s = round(time.time() - started)
tries = run.stderr.count('open model reply:')  # pack.py logs one line per Glimmer reply; each retry adds one
marks['letter_written'] = round(time.time() - t0)
print(run, f'{pack_s} s, {tries} replies from Glimmer ({max(tries - 1, 0)} retries)', sep='\n')
if run.returncode or not os.path.exists(f'{OUT}/plan.json'):
    raise RuntimeError('Glimmer wrote no letter:\n' + run.stderr[-3000:])
# saved outputs: the letter and its plan, about sample/ (made-up tracks, no video), so safe to publish
shutil.copyfile(f'{OUT}/plan.json', '/kaggle/working/plan.json')
with open('/kaggle/working/glimmer-letter.txt', 'w') as f:
    f.write(json.load(open(f'{OUT}/plan.json'))['letter'] + '\n')

In [ ]:
from IPython.display import HTML
HTML(open(f'{OUT}/index.html').read())

## 5. Run record: `glimmer-gate-provenance.json`

What ran (llama.cpp build, model revision and hashes, the git commit and pack.py's sha256), on what (nvidia-smi), how
long each step took, how many tries Glimmer needed, and the letter's sha256. Download it from the output and commit it as `kaggle/glimmer-gate-provenance.json`.

In [ ]:
plan = json.load(open(f'{OUT}/plan.json'))
settings = PROPS.get('default_generation_settings') or {}
provenance = {
    'team': 'Run by the author on the author\'s own Kaggle account (free 2 x T4).',
    'notebook': 'kaggle/glimmer-gate.ipynb, on our gate (cw-gate)',
    'code': {'git_sha': open(f'{CW}/GIT_SHA').read().strip() if os.path.exists(f'{CW}/GIT_SHA') else None,
             'pack_py_sha256': sha256(f'{CW}/pack.py')},
    'nvidia_smi': NVIDIA_SMI,
    'llama_cpp': {'build_info': PROPS.get('build_info'), 'args': ARGS + ['-c', str(CONTEXT)],
                  'n_ctx_slot': settings.get('n_ctx'),
                  'sampling': {k: (settings.get('params') or {}).get(k) for k in ('temperature', 'top_k', 'top_p', 'min_p')}},
    'model': {'repo': REPO, 'revision': REVISION, 'sha256_checked': SHA256},
    'elapsed_s': marks,  # each step's end, from the first cell
    'pack_py_s': pack_s,
    'street_photo_sent': os.path.exists(f'{OUT}/street.jpg'),
    'letter': {'writer': plan.get('writer'), 'fix': plan.get('fix'),
               'sha256': hashlib.sha256(plan['letter'].encode()).hexdigest(),
               'timings': plan.get('timings'), 'usage': plan.get('usage'), 'judged_by': plan.get('judged_by'),
               'tries': tries,  # replies from Glimmer: 1 means no retry
               'unmeasured': plan.get('unmeasured', []), 'unsupported': plan.get('unsupported', [])},
}
with open('/kaggle/working/glimmer-gate-provenance.json', 'w') as f:
    json.dump(provenance, f, indent=1, ensure_ascii=False)
print(json.dumps(provenance, indent=1, ensure_ascii=False))